In [28]:
import ollama
import numpy as np
from IPython.display import display, Markdown
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os

In [5]:
with open("sample.txt", "r") as file:
    text = file.read()

# chunking 
chunks = text.split(".")
new_chunk = []

for i in range(len(chunks)):
    temp = chunks[i].replace('\n',' ')
    if len(temp.strip())>0:
        new_chunk.append(temp)

chunks = new_chunk 


new_chunk = []
window_size = 3
for i in range(len(chunks)-window_size+1):
    new_chunk.append("".join([chunks[i+j] for j in range(window_size)]))
sentence_chunks = chunks 
chunks = new_chunk

In [8]:
sentence_chunks 

['Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation',
 '  It allows language models to access external knowledge sources to improve factual accuracy',
 '  Chunking plays a critical role in RAG systems',
 ' If chunks are too large, embeddings may lose specificity',
 '  If chunks are too small, important context may be lost',
 '  Vector databases store embeddings and allow similarity-based search',
 ' Common similarity measures include cosine similarity',
 '  The quality of retrieval directly affects the final generated answer',
 ' Poor retrieval leads to hallucinations,  even if the language model itself is strong']

In [9]:
#embedding
vector_list = [ollama.embeddings(prompt = i, model="mxbai-embed-large") for i in chunks]
vector_db = np.array([np.array(i['embedding']) for i in vector_list])

In [10]:
def get_embedding(prompt):
    return np.array(ollama.embeddings(prompt = prompt, model="mxbai-embed-large")['embedding'])

In [11]:
def cosine_similarity(a,b):
    dot_product = np.dot(a,b) 
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

In [12]:
def get_relevant_chunk_index(query_embedding , k = len(vector_db)):
    similarity_vector = []
    top_k = []
    for i in vector_db:
        similarity_vector.append(cosine_similarity(query_embedding,i))
    top_k =  [index for index ,value in sorted(list(enumerate(similarity_vector)),key= lambda x : x[1],reverse= True)[:k]]
    return top_k
    

In [13]:
def LLM_call(prompt,model):
    return ollama.generate(
        model = model,
        prompt = prompt
    )['response']


In [14]:
prompt_template = """
        You are an helpful assistant. 
        You are good with framing human like informative and short answers based on given context.
        Generate response after thorough understanding of the context in simple and polite way.
        Use the technical wording as per context as and when required as per question.
        If the answer is not present in context just repond "I don't know".

        context : {context}

        question : {user_query}
"""
def rag(user_query):
    query_embedding = get_embedding(user_query)
    top_k_chunk = get_relevant_chunk_index(query_embedding, window_size)

    unique_top_k_chunk = []
    for index in top_k_chunk :
        unique_top_k_chunk.extend([i for i in range(index,index+window_size)])
    unique_top_k_chunk  = set(unique_top_k_chunk)
    context = "".join([sentence_chunks[i] for i in unique_top_k_chunk])

    prompt = prompt_template.format(context = context, user_query = user_query)
    response = LLM_call(prompt = prompt ,model = "gemma4")

    return response,context

In [15]:
questions = ['What is RAG?','Why does chunk size matter in RAG?','What similarity measure is commonly used?','Does RAG eliminate hallucinations completely?',"Who is world's richest man?"]

responses = []
contexts = []

for i in questions:
    response, context = rag(i)
    responses.append(response)
    contexts.append(context)

In [17]:
responses

['RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It allows language models to access external knowledge sources, which helps improve their factual accuracy.',
 'Chunk size is critical in Retrieval-Augmented Generation (RAG) systems because it affects the quality of the retrieved information.\n\n*   If the chunks are too large, the embeddings may lose specificity.\n*   Conversely, if the chunks are too small, important context may be lost.',
 'The context indicates that a common similarity measure used is cosine similarity.',
 'Based on the context, RAG does not guarantee the complete elimination of hallucinations. The quality of retrieval directly affects the final generated answer, and poor retrieval can still lead to hallucinations, even when the language model itself is strong.',
 "I don't know"]

In [18]:
contexts


['Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation  It allows language models to access external knowledge sources to improve factual accuracy  Chunking plays a critical role in RAG systems If chunks are too large, embeddings may lose specificity  If chunks are too small, important context may be lost',
 '  It allows language models to access external knowledge sources to improve factual accuracy  Chunking plays a critical role in RAG systems If chunks are too large, embeddings may lose specificity  If chunks are too small, important context may be lost  Vector databases store embeddings and allow similarity-based search',
 '  If chunks are too small, important context may be lost  Vector databases store embeddings and allow similarity-based search Common similarity measures include cosine similarity  The quality of retrieval directly affects the final generated answer Poor retrieval leads to hallucinations,  even if the langu